# CUDA-Only Unsloth Banking77 Generalization Training

This notebook fine-tunes an Unsloth LoRA model on [`tsilva/banking77`](https://huggingface.co/datasets/tsilva/banking77) for intent classification. The goal is no longer memorization; the goal is **test-set generalization**.

The data contract is:

- `system_prompt` -> system message
- `text` -> user prompt
- `label_text` -> assistant response

The training flow is:

- stratified train/validation split from the Hugging Face `train` split
- supervised fine-tuning with `SFTTrainer`
- early stopping on validation loss
- best-checkpoint restore
- deterministic generation on the Hugging Face `test` split
- exact-match test accuracy and W&B prediction logging

It is intentionally **CUDA-only**. If CUDA, Unsloth, TRL, W&B, or the model stack is missing, the notebook stops instead of falling back to CPU, MPS, or a toy local model.


## 1. Environment Contract

Run this in a Linux or Windows CUDA environment with Unsloth installed. The repo's Modal runner is the intended validation path for this notebook.

```bash
./run_modal.sh notebooks/unsloth-minimal-training.ipynb --gpu-type L40S
```

W&B logging expects the Modal secret `wandb-secret` to provide `WANDB_API_KEY`.


### Configuration

The defaults are tuned for fast L40S iteration. `run_test_evaluation=False` keeps the full Banking77 test split reserved for finalist runs.


In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 3407,

    # Data
    'dataset_name': 'tsilva/banking77',
    'train_split': 'train',
    'test_split': 'test',
    'val_fraction': 0.1,
    'train_limit': None,  # Set to an int for quick debugging; None uses the full train split
    'test_eval_limit': None,  # None evaluates all 3,076 test rows

    # Model
    'model_name': 'unsloth/gemma-3-1b-it-unsloth-bnb-4bit',
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'load_in_8bit': False,
    'full_finetuning': False,

    # LoRA
    'lora_rank': 32,
    'lora_alpha': 32,
    'lora_dropout': 0,

    # Training
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'gradient_accumulation_steps': 1,
    'max_steps': 2000,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.02,
    'lr_scheduler_type': 'cosine',
    'logging_steps': 25,
    'eval_steps': 400,
    'early_stopping_patience': 2,
    'early_stopping_threshold': 1e-4,
    'group_by_length': True,
    'throughput_token_sample_size': 512,  # Rows sampled to estimate training tokens/sec
    'run_validation_generation': False,  # Generation metrics are expensive; enable for finalist runs
    'val_generation_examples_per_label': 1,  # Stratified validation rows generated when enabled
    'val_generation_log_examples': 16,  # Example rows to keep in each W&B validation table
    'run_test_evaluation': False,  # Keep test set untouched until a finalist run
    'output_dir': 'outputs/unsloth-banking77-generalization',

    # Experiment tracking
    'wandb_project': 'unsloth-minimal-training',
    'wandb_run_name': 'unsloth-banking77-l40s-fast',
    'wandb_run_id': 'banking77-l40s-fast-20260421-162516',
    'target_gpu_name': 'Nvidia L40S',
    'target_gpu_hourly_cost_usd': 1.9512,  # Modal L40S: $0.000542/sec

    # Inference
    'max_new_tokens': 16,
    'generation_eval_batch_size': 64,
}


### Hard CUDA and Package Checks

The shared helper prints package, Python, PyTorch, CUDA, cuDNN, and GPU details, then stops early if the runtime cannot support this notebook.


In [ ]:
import torch

from aiml_notebooks import require_environment

ENVIRONMENT = require_environment(
    packages=['unsloth', 'trl', 'datasets', 'transformers', 'bitsandbytes', 'wandb'],
    cuda=True,
    min_cuda_capability=(7, 0),
    error_prefix='CUDA fine-tuning environment check failed',
)


### Import Unsloth Primitives

Importing Unsloth before other model libraries lets it patch kernels and model classes as intended.


In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTConfig, SFTTrainer


## 2. Load Banking77 and Build Splits

The Hugging Face `train` split is split again into train and validation partitions using a deterministic stratified split over the integer label column. The original Hugging Face `test` split is kept untouched for final accuracy.


### Load the Raw Dataset

Start by fetching the Hugging Face dataset and naming the source splits we will use later.


In [ ]:
import random
from collections import defaultdict

from datasets import load_dataset

raw_dataset = load_dataset(CONFIG['dataset_name'])
source_train = raw_dataset[CONFIG['train_split']]
test_dataset = raw_dataset[CONFIG['test_split']]

print(raw_dataset)
print(f"Source train rows: {len(source_train):,}")
print(f"Source test rows: {len(test_dataset):,}")


### Group Training Rows by Label

Stratification means every label contributes roughly the same validation fraction.


In [ ]:
label_to_indices = defaultdict(list)
for index, label in enumerate(source_train['label']):
    label_to_indices[int(label)].append(index)

class_sizes = [len(indices) for indices in label_to_indices.values()]
print(f"Labels found: {len(label_to_indices)}")
print(f"Smallest class size: {min(class_sizes):,}")
print(f"Largest class size: {max(class_sizes):,}")


### Draw Stratified Train and Validation Indices

For each label, shuffle only that label's rows, then move a fixed fraction into validation.


In [ ]:
rng = random.Random(CONFIG['seed'])
train_indices = []
val_indices = []
for label, indices in sorted(label_to_indices.items()):
    shuffled = list(indices)
    rng.shuffle(shuffled)
    val_count = max(1, round(len(shuffled) * CONFIG['val_fraction']))
    val_indices.extend(shuffled[:val_count])
    train_indices.extend(shuffled[val_count:])

rng.shuffle(train_indices)
rng.shuffle(val_indices)

print(f"Train indices before limit: {len(train_indices):,}")
print(f"Validation indices: {len(val_indices):,}")
print(f"Validation fraction: {len(val_indices) / len(source_train):.1%}")


### Apply Optional Debug Limits

The training limit is useful for smoke tests. The test limit keeps final evaluation cheap when requested.


In [ ]:
if CONFIG['train_limit'] is not None:
    train_indices = train_indices[:CONFIG['train_limit']]

train_dataset = source_train.select(train_indices)
val_dataset = source_train.select(val_indices)
if CONFIG['test_eval_limit'] is not None:
    test_dataset = test_dataset.select(range(CONFIG['test_eval_limit']))

print(f"Train rows: {len(train_dataset):,}")
print(f"Validation rows: {len(val_dataset):,}")
print(f"Test rows: {len(test_dataset):,}")


### Capture the Valid Label Vocabulary

Later, generated text is normalized back to one of these intent labels before scoring.


In [ ]:
LABELS = sorted(set(source_train['label_text']))
LABEL_SET = set(LABELS)

print(f"Labels: {len(LABELS)}")
print("First 10 labels:")
for label in LABELS[:10]:
    print(f"- {label}")


### Inspect One Training Example

A concrete row shows the exact fields that become the chat conversation.


In [ ]:
train_dataset[0]


### Normalize Generated Labels

The model may produce a label plus punctuation or extra words. This helper keeps scoring deterministic.


In [ ]:
def normalize_label(text: str) -> str:
    cleaned = text.strip().strip('`').strip().strip('.,;:!')
    first_token = cleaned.split()[0] if cleaned.split() else ''
    first_token = first_token.strip('`').strip().strip('.,;:!')
    if cleaned in LABEL_SET:
        return cleaned
    if first_token in LABEL_SET:
        return first_token
    return first_token


for example in ['cash_withdrawal', '`cash_withdrawal`.', 'cash_withdrawal please']:
    print(f"{example!r} -> {normalize_label(example)!r}")


## 3. Load a Real Unsloth Model

The default model is Gemma 3 1B Instruct in Unsloth 4-bit form. That is compact enough for fast L40S runs, but more capable than the 270M smoke-test model.


In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    load_in_4bit=CONFIG['load_in_4bit'],
    load_in_8bit=CONFIG['load_in_8bit'],
    full_finetuning=CONFIG['full_finetuning'],
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='gemma3',
)


### Attach LoRA Adapters

This is the Unsloth adapter primitive. The base model stays frozen while selected projection modules receive trainable low-rank matrices.


In [ ]:
model = FastModel.get_peft_model(
    model,
    r=CONFIG['lora_rank'],
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=CONFIG['seed'],
    use_rslora=False,
    loftq_config=None,
)


### Inspect Trainable Parameters

This confirms we are training a LoRA adapter instead of full fine-tuning the base model.


In [ ]:
trainable_params = 0
total_params = 0
for param in model.parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f'Trainable parameters: {trainable_params:,}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable fraction: {100 * trainable_params / total_params:.3f}%')
assert 0 < trainable_params < total_params


## 4. Format the Dataset with the Chat Template

Every row becomes a three-message conversation: system instructions, user text, and assistant intent label.


In [ ]:
def to_messages(row):
    return {
        'messages': [
            {'role': 'system', 'content': row['system_prompt']},
            {'role': 'user', 'content': row['text']},
            {'role': 'assistant', 'content': row['label_text']},
        ]
    }


### Convert Rows to Chat Messages

This makes the structure explicit before the tokenizer turns it into model text.


In [ ]:
message_train_dataset = train_dataset.map(to_messages)
message_val_dataset = val_dataset.map(to_messages)
message_train_dataset[0]['messages']


### Define the Chat Template Renderer

`apply_chat_template` adds the model-specific turn markers expected by Gemma 3.


In [ ]:
def apply_template(row):
    return {
        'text': tokenizer.apply_chat_template(
            row['messages'],
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix('<bos>')
    }


### Render the Training and Validation Text

The `text` field is what `SFTTrainer` reads during supervised fine-tuning.


In [ ]:
formatted_train_dataset = message_train_dataset.map(apply_template)
formatted_val_dataset = message_val_dataset.map(apply_template)

print(f"Formatted train rows: {len(formatted_train_dataset):,}")
print(f"Formatted validation rows: {len(formatted_val_dataset):,}")
print(formatted_train_dataset[0]['text'][:1200])


## 5. Build the SFT Trainer with Early Stopping

Validation loss is evaluated every `eval_steps`. Early stopping watches that validation loss and `load_best_model_at_end=True` restores the best checkpoint before final test evaluation.


In [ ]:
import os
import time

import wandb
from transformers import EarlyStoppingCallback, TrainerCallback

os.environ['WANDB_PROJECT'] = CONFIG['wandb_project']
os.environ.setdefault('WANDB_LOG_MODEL', 'false')

print(f"W&B project: {os.environ['WANDB_PROJECT']}")
print(f"W&B model artifact logging: {os.environ['WANDB_LOG_MODEL']}")


### Estimate Tokens per Training Sample

A small tokenized sample gives us a practical throughput denominator before training starts.


In [ ]:
throughput_sample_size = min(CONFIG['throughput_token_sample_size'], len(formatted_train_dataset))
throughput_texts = list(formatted_train_dataset.select(range(throughput_sample_size))['text'])
throughput_token_lengths = [
    len(input_ids)
    for input_ids in tokenizer(
        throughput_texts,
        add_special_tokens=False,
        truncation=True,
        max_length=CONFIG['max_seq_length'],
    )['input_ids']
]
avg_train_tokens_per_sample = sum(throughput_token_lengths) / len(throughput_token_lengths)

print(f"Tokenized rows: {throughput_sample_size:,}")
print(f"Average train tokens/sample: {avg_train_tokens_per_sample:.1f}")


### Compute Effective Batch and Token Counts

Gradient accumulation means one optimizer update sees multiple microbatches.


In [ ]:
effective_batch_size = (
    CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
)
effective_tokens_per_step = effective_batch_size * avg_train_tokens_per_sample

print(f"Effective batch size: {effective_batch_size:,}")
print(f"Effective tokens/optimizer step: {effective_tokens_per_step:.1f}")


### Inspect the GPU Budget

We log the actual GPU so W&B charts can be compared against the intended target machine.


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
gpu_total_memory_gb = gpu_stats.total_memory / 1024**3

print(f"GPU: {gpu_stats.name}")
print(f"Total GPU memory: {gpu_total_memory_gb:.2f} GB")
print(f"Target GPU: {CONFIG['target_gpu_name']}")


### Define a Stratified Validation Sampler

Generation-based validation is expensive, so we sample a few examples per label.


In [ ]:
def select_stratified_rows(dataset, examples_per_label, seed):
    rows_by_label = defaultdict(list)
    for row in dataset:
        rows_by_label[row['label_text']].append(row)

    rng = random.Random(seed)
    selected_rows = []
    for label in sorted(rows_by_label):
        label_rows = rows_by_label[label]
        rng.shuffle(label_rows)
        selected_rows.extend(label_rows[:examples_per_label])
    rng.shuffle(selected_rows)
    return selected_rows


### Select Validation Rows for Generation Metrics

These rows will be generated at each evaluation step and logged to W&B.


In [ ]:
validation_generation_rows = select_stratified_rows(
    val_dataset,
    examples_per_label=(
        CONFIG['val_generation_examples_per_label']
        if CONFIG['run_validation_generation']
        else 0
    ),
    seed=CONFIG['seed'],
)

print(f"Validation generation rows: {len(validation_generation_rows):,}")
if validation_generation_rows:
    print(validation_generation_rows[0])
else:
    print("Validation generation is disabled for this fast training profile.")


### Define Deterministic Label Generation

During validation and final testing, the model receives only the system and user turns, then generates the label. Batching keeps optional generation metrics practical on L40S.


In [ ]:
def generation_prompt_text(tokenizer, row) -> str:
    messages = [
        {'role': 'system', 'content': row['system_prompt']},
        {'role': 'user', 'content': row['text']},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    ).removeprefix('<bos>')


def row_batch(rows, start, end):
    return [rows[index] for index in range(start, end)]


@torch.inference_mode()
def generate_label_batch(model, tokenizer, rows) -> list[str]:
    if not rows:
        return []

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    texts = [generation_prompt_text(tokenizer, row) for row in rows]
    try:
        inputs = tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=CONFIG['max_seq_length'],
        ).to('cuda')
    finally:
        tokenizer.padding_side = original_padding_side

    prompt_width = inputs['input_ids'].shape[-1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=CONFIG['max_new_tokens'],
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = outputs[:, prompt_width:]
    return [
        prediction.strip()
        for prediction in tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    ]


def generate_labels(model, tokenizer, rows, batch_size) -> list[str]:
    predictions = []
    for start in range(0, len(rows), batch_size):
        end = min(start + batch_size, len(rows))
        predictions.extend(generate_label_batch(model, tokenizer, row_batch(rows, start, end)))
    return predictions


def generate_label(model, tokenizer, row) -> str:
    return generate_label_batch(model, tokenizer, [row])[0]


### Define Macro F1

Accuracy can hide weak labels, so macro F1 gives each intent equal weight.


In [ ]:
def macro_f1_score(expected_labels, predicted_labels):
    f1_scores = []
    for label in LABELS:
        true_positive = sum(
            expected == label and predicted == label
            for expected, predicted in zip(expected_labels, predicted_labels)
        )
        false_positive = sum(
            expected != label and predicted == label
            for expected, predicted in zip(expected_labels, predicted_labels)
        )
        false_negative = sum(
            expected == label and predicted != label
            for expected, predicted in zip(expected_labels, predicted_labels)
        )

        precision_denominator = true_positive + false_positive
        recall_denominator = true_positive + false_negative
        precision = true_positive / precision_denominator if precision_denominator else 0.0
        recall = true_positive / recall_denominator if recall_denominator else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        f1_scores.append(f1)
    return sum(f1_scores) / len(f1_scores)


print(f"Perfect macro F1 smoke test: {macro_f1_score(LABELS[:3], LABELS[:3]):.1f}")


### Start the W&B Run

A timestamped run name keeps repeated notebook executions separate.


In [ ]:
wandb_run_name = f"{CONFIG['wandb_run_name']}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
wandb_run = wandb.init(
    project=CONFIG['wandb_project'],
    id=CONFIG['wandb_run_id'],
    name=wandb_run_name,
    config=CONFIG,
    resume='allow',
)
wandb_run.config.update(CONFIG, allow_val_change=True)

wandb_run_id = wandb_run.id
print(f"W&B run: https://wandb.ai/tsilva/{CONFIG['wandb_project']}/runs/{wandb_run_id}")


### Log Run Setup Metadata

These values explain the scale, hardware, and validation sample attached to this run.


In [ ]:
wandb_run.config.update(
    {
        'wandb_actual_run_name': wandb_run_name,
        'throughput/avg_train_tokens_per_sample': avg_train_tokens_per_sample,
        'throughput/token_sample_size': throughput_sample_size,
        'throughput/effective_tokens_per_step': effective_tokens_per_step,
        'train/effective_batch_size': effective_batch_size,
        'hardware/gpu_name': gpu_stats.name,
        'hardware/gpu_total_memory_gb': gpu_total_memory_gb,
        'hardware/target_gpu_name': CONFIG['target_gpu_name'],
        'cost/gpu_hourly_cost_usd': CONFIG['target_gpu_hourly_cost_usd'],
        'val/generation_total': len(validation_generation_rows),
    },
    allow_val_change=True,
)

print(f"Run name: {wandb_run_name}")
print(f"Validation generation rows per eval: {len(validation_generation_rows)}")


### Normalize Trainer Log Names

The Hugging Face trainer emits mixed metric names. This helper puts them into stable W&B namespaces.


In [ ]:
def namespace_trainer_logs(logs):
    metric_name_map = {
        'loss': 'train/loss',
        'grad_norm': 'train/grad_norm',
        'learning_rate': 'train/lr',
        'epoch': 'train/epoch',
        'eval_loss': 'val/loss',
        'eval_runtime': 'val/runtime',
        'eval_samples_per_second': 'val/samples_per_second',
        'eval_steps_per_second': 'val/steps_per_second',
    }
    numeric_logs = {}
    for key, value in logs.items():
        if not isinstance(value, (int, float)):
            continue
        if key in metric_name_map:
            numeric_logs[metric_name_map[key]] = value
        elif '/' in key:
            numeric_logs[key] = value
        else:
            numeric_logs[f'trainer/{key}'] = value
    return numeric_logs


### Compute Trainer Progress Metrics

These scalars make the progress bar visible in W&B even when the notebook is not open.


In [ ]:
def trainer_progress_metrics(args, state, train_start_time=None):
    max_steps = int(getattr(state, 'max_steps', 0) or getattr(args, 'max_steps', 0) or 0)
    max_epochs = getattr(state, 'num_train_epochs', None)
    if max_epochs is None:
        max_epochs = getattr(args, 'num_train_epochs', None)
    current_epoch = getattr(state, 'epoch', None)
    metrics = {'train/global_step': state.global_step}
    if max_steps > 0:
        metrics['train/max_steps'] = max_steps
        metrics['train/progress_percent'] = 100 * state.global_step / max_steps
    if isinstance(max_epochs, (int, float)) and max_epochs > 0:
        metrics['train/max_epochs'] = max_epochs
    if isinstance(current_epoch, (int, float)):
        metrics['train/epoch'] = current_epoch

    if train_start_time is None:
        return metrics

    elapsed_seconds = time.perf_counter() - train_start_time
    progress_units = None
    target_units = None
    if max_steps > 0:
        progress_units = state.global_step
        target_units = max_steps
        metrics['train/eta_stop_by_steps'] = 1
        metrics['train/eta_stop_by_epochs'] = 0
    elif (
        isinstance(current_epoch, (int, float))
        and isinstance(max_epochs, (int, float))
        and max_epochs > 0
    ):
        progress_units = current_epoch
        target_units = max_epochs
        metrics['train/eta_stop_by_steps'] = 0
        metrics['train/eta_stop_by_epochs'] = 1

    if (
        progress_units is not None
        and target_units is not None
        and progress_units > 0
        and elapsed_seconds > 0
    ):
        units_per_second = progress_units / elapsed_seconds
        remaining_units = max(target_units - progress_units, 0)
        eta_seconds = remaining_units / units_per_second if units_per_second > 0 else 0.0
        total_runtime_seconds = elapsed_seconds + eta_seconds
        metrics.update(
            {
                'train/eta_seconds': eta_seconds,
                'train/eta_minutes': eta_seconds / 60,
                'train/eta_hours': eta_seconds / 3600,
                'train/estimated_total_runtime_hours': total_runtime_seconds / 3600,
                'train/remaining_progress_units': remaining_units,
            }
        )
    return metrics


### Compute Training Throughput Metrics

Throughput converts elapsed step time into steps, samples, tokens, and tokens per dollar.


In [ ]:
def training_throughput_metrics(
    args,
    accumulated_steps,
    accumulated_seconds,
    avg_tokens_per_sample,
    gpu_hourly_cost_usd,
):
    world_size = max(1, int(getattr(args, 'world_size', 1) or 1))
    effective_batch_size = (
        args.per_device_train_batch_size
        * args.gradient_accumulation_steps
        * world_size
    )
    train_steps_per_second = accumulated_steps / accumulated_seconds
    train_samples_per_second = accumulated_steps * effective_batch_size / accumulated_seconds
    train_tokens_per_second = train_samples_per_second * avg_tokens_per_sample
    price_per_second = gpu_hourly_cost_usd / 3600

    metrics = {
        'train/steps_per_second': train_steps_per_second,
        'train/samples_per_second': train_samples_per_second,
        'train/tokens_per_second': train_tokens_per_second,
        'train/seconds_per_step': accumulated_seconds / accumulated_steps,
        'train/effective_batch_size': effective_batch_size,
        'throughput/effective_tokens_per_step': (
            effective_batch_size * avg_tokens_per_sample
        ),
    }
    if price_per_second > 0:
        metrics['efficiency/train_tokens_per_dollar'] = (
            train_tokens_per_second / price_per_second
        )
    return metrics


### Compute CUDA Memory Metrics

These values answer the practical question: did the run fit comfortably on this GPU?


In [ ]:
def cuda_memory_metrics(gpu_total_memory_gb):
    reserved_gb = torch.cuda.memory_reserved() / 1024**3
    allocated_gb = torch.cuda.memory_allocated() / 1024**3
    peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
    peak_reserved_percent = 100 * peak_reserved_gb / gpu_total_memory_gb
    return {
        'device_fit/vram_reserved_gb': reserved_gb,
        'device_fit/vram_allocated_gb': allocated_gb,
        'device_fit/peak_vram_reserved_gb': peak_reserved_gb,
        'device_fit/peak_vram_reserved_percent': peak_reserved_percent,
        'device_fit/vram_headroom_percent': 100 - peak_reserved_percent,
    }


### Compute Training Cost Metrics

Cost is estimated from wall time and the configured target GPU hourly price.


In [ ]:
def training_cost_metrics(train_start_time, gpu_hourly_cost_usd):
    elapsed_gpu_hours = (time.perf_counter() - train_start_time) / 3600
    return {
        'cost/elapsed_gpu_hours': elapsed_gpu_hours,
        'cost/estimated_usd': elapsed_gpu_hours * gpu_hourly_cost_usd,
    }


### Compute Loss-Step Timing Metrics

Loss logs are the cadence where we convert accumulated step timing into throughput.


In [ ]:
def loss_step_timing_metrics(
    args,
    state,
    last_train_log_step,
    last_train_log_wall_time,
    accumulated_train_steps,
    accumulated_train_step_seconds,
    avg_tokens_per_sample,
    gpu_hourly_cost_usd,
):
    now = time.perf_counter()
    metrics = {}
    reset_accumulators = False

    if last_train_log_wall_time is None or last_train_log_step is None:
        return metrics, now, reset_accumulators

    step_delta = state.global_step - last_train_log_step
    wall_elapsed_seconds = now - last_train_log_wall_time
    if accumulated_train_steps > 0 and accumulated_train_step_seconds > 0:
        metrics.update(
            training_throughput_metrics(
                args,
                accumulated_train_steps,
                accumulated_train_step_seconds,
                avg_tokens_per_sample,
                gpu_hourly_cost_usd,
            )
        )
        reset_accumulators = True
    if step_delta > 0 and wall_elapsed_seconds > 0:
        metrics['train/wall_steps_per_second'] = step_delta / wall_elapsed_seconds
    return metrics, now, reset_accumulators


### Define the Scalar W&B Callback

This callback turns trainer logs into progress, throughput, memory, and cost metrics.


In [ ]:
class WandbScalarLogger(TrainerCallback):
    def __init__(self, avg_tokens_per_sample, gpu_total_memory_gb, gpu_hourly_cost_usd):
        self.avg_tokens_per_sample = avg_tokens_per_sample
        self.last_train_log_step = None
        self.last_train_log_wall_time = None
        self.current_step_start_time = None
        self.accumulated_train_step_seconds = 0.0
        self.accumulated_train_steps = 0
        self.gpu_total_memory_gb = gpu_total_memory_gb
        self.gpu_hourly_cost_usd = gpu_hourly_cost_usd
        self.train_start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.last_train_log_step = state.global_step
        self.last_train_log_wall_time = time.perf_counter()
        self.train_start_time = self.last_train_log_wall_time

    def on_step_begin(self, args, state, control, **kwargs):
        self.current_step_start_time = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self.current_step_start_time is None:
            return
        self.accumulated_train_step_seconds += time.perf_counter() - self.current_step_start_time
        self.accumulated_train_steps += 1
        self.current_step_start_time = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or wandb.run is None:
            return
        numeric_logs = namespace_trainer_logs(logs)
        numeric_logs.update(trainer_progress_metrics(args, state, self.train_start_time))

        if 'loss' in logs:
            timing_metrics, now, reset_accumulators = loss_step_timing_metrics(
                args,
                state,
                self.last_train_log_step,
                self.last_train_log_wall_time,
                self.accumulated_train_steps,
                self.accumulated_train_step_seconds,
                self.avg_tokens_per_sample,
                self.gpu_hourly_cost_usd,
            )
            numeric_logs.update(timing_metrics)
            if reset_accumulators:
                self.accumulated_train_step_seconds = 0.0
                self.accumulated_train_steps = 0
            self.last_train_log_step = state.global_step
            self.last_train_log_wall_time = now

        if torch.cuda.is_available():
            numeric_logs.update(cuda_memory_metrics(self.gpu_total_memory_gb))

        if self.train_start_time is not None:
            numeric_logs.update(
                training_cost_metrics(self.train_start_time, self.gpu_hourly_cost_usd)
            )

        if numeric_logs:
            wandb.log(numeric_logs, step=state.global_step)


### Score Validation Generations

This loop turns validation prompts into predictions and saves a small inspectable table sample.


In [ ]:
def score_validation_generations(model, tokenizer, rows, max_logged_examples, global_step):
    expected_labels = []
    predicted_labels = []
    example_rows = []
    predictions = generate_labels(
        model,
        tokenizer,
        rows,
        CONFIG['generation_eval_batch_size'],
    )

    for index, (row, prediction) in enumerate(zip(rows, predictions)):
        normalized_prediction = normalize_label(prediction)
        expected = row['label_text']
        expected_labels.append(expected)
        predicted_labels.append(normalized_prediction)
        if len(example_rows) < max_logged_examples:
            example_rows.append(
                [
                    global_step,
                    index,
                    row['text'],
                    expected,
                    prediction,
                    normalized_prediction,
                    normalized_prediction == expected,
                ]
            )

    return expected_labels, predicted_labels, example_rows


### Summarize Validation Generation Metrics

The generated labels become accuracy, macro F1, out-of-label rate, speed, and cost metrics.


In [ ]:
def validation_generation_metrics(
    expected_labels,
    predicted_labels,
    generation_runtime,
    train_start_time,
    gpu_hourly_cost_usd,
):
    total = len(expected_labels)
    correct = sum(
        expected == predicted
        for expected, predicted in zip(expected_labels, predicted_labels)
    )
    accuracy = correct / total if total else 0.0
    macro_f1 = macro_f1_score(expected_labels, predicted_labels) if total else 0.0
    out_of_label_predictions = sum(
        predicted not in LABEL_SET
        for predicted in predicted_labels
    )

    logs = {
        'val/acc': accuracy,
        'val/macro_f1': macro_f1,
        'val/correct': correct,
        'val/total': total,
        'val/out_of_label_rate': out_of_label_predictions / total if total else 0.0,
        'val/generation_runtime': generation_runtime,
        'val/generation_samples_per_second': total / generation_runtime
        if generation_runtime > 0
        else 0.0,
    }
    if train_start_time is not None:
        estimated_cost = (
            (time.perf_counter() - train_start_time)
            / 3600
            * gpu_hourly_cost_usd
        )
        logs['cost/estimated_usd'] = estimated_cost
        if estimated_cost > 0:
            logs['efficiency/val_acc_per_usd'] = accuracy / estimated_cost
            logs['efficiency/val_macro_f1_per_usd'] = macro_f1 / estimated_cost
    return logs


### Log Validation Prediction Tables

W&B tables preserve examples so metric changes can be traced back to real prompts.


In [ ]:
def log_validation_prediction_table(example_rows, global_step):
    if not example_rows:
        return
    wandb.log(
        {
            'val/generation_predictions': wandb.Table(
                columns=[
                    'step',
                    'index',
                    'text',
                    'expected_response',
                    'prediction',
                    'normalized_prediction',
                    'exact_match',
                ],
                data=example_rows,
            )
        },
        step=global_step,
    )


### Define the Validation Generation Callback

This callback runs deterministic generation on the stratified validation sample after each evaluation.


In [ ]:
class ValidationGenerationLogger(TrainerCallback):
    def __init__(self, rows, tokenizer, max_logged_examples, gpu_hourly_cost_usd):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_logged_examples = max_logged_examples
        self.gpu_hourly_cost_usd = gpu_hourly_cost_usd
        self.train_start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start_time = time.perf_counter()

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None or wandb.run is None:
            return

        started_at = time.perf_counter()
        was_training = model.training
        model.eval()
        expected_labels, predicted_labels, example_rows = score_validation_generations(
            model,
            self.tokenizer,
            self.rows,
            self.max_logged_examples,
            state.global_step,
        )

        if was_training:
            model.train()

        generation_runtime = time.perf_counter() - started_at
        logs = validation_generation_metrics(
            expected_labels,
            predicted_labels,
            generation_runtime,
            self.train_start_time,
            self.gpu_hourly_cost_usd,
        )
        wandb.log(logs, step=state.global_step)
        log_validation_prediction_table(example_rows, state.global_step)


### Track Early-Stopping Patience

This callback mirrors the early-stopping decision rule so W&B can show how close a run is to stopping.


In [ ]:
class EarlyStoppingPatienceLogger(TrainerCallback):
    def __init__(self, patience, threshold=0.0):
        self.patience = patience
        self.threshold = threshold
        self.best_eval_loss = None
        self.bad_eval_count = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None or wandb.run is None:
            return

        eval_loss = metrics.get('eval_loss')
        if not isinstance(eval_loss, (int, float)):
            return

        improved = (
            self.best_eval_loss is None
            or eval_loss < self.best_eval_loss - self.threshold
        )
        if improved:
            self.best_eval_loss = eval_loss
            self.bad_eval_count = 0
        else:
            self.bad_eval_count += 1

        wandb.log(
            {
                'early_stop/patience_remaining': max(
                    self.patience - self.bad_eval_count,
                    0,
                ),
                'early_stop/patience_used': self.bad_eval_count,
                'early_stop/patience': self.patience,
                'early_stop/best_val_loss': self.best_eval_loss,
                'early_stop/improved': int(improved),
            },
            step=state.global_step,
        )


### Configure SFT Training Arguments

These settings control optimization, evaluation cadence, checkpointing, and best-model restore.


In [ ]:
training_args = SFTConfig(
    dataset_text_field='text',
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    max_steps=CONFIG['max_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    logging_steps=CONFIG['logging_steps'],
    eval_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_strategy='steps',
    save_steps=CONFIG['eval_steps'],
    save_total_limit=2,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    load_best_model_at_end=True,
    optim='adamw_8bit',
    weight_decay=0.001,
    lr_scheduler_type=CONFIG['lr_scheduler_type'],
    seed=CONFIG['seed'],
    output_dir=CONFIG['output_dir'],
    report_to='none',
    run_name=wandb_run_name,
)

print(f"Max steps: {training_args.max_steps:,}")
print(f"Evaluate every {training_args.eval_steps:,} steps")
print(f"Best-model metric: {training_args.metric_for_best_model}")


### Assemble Trainer Callbacks

The callbacks add W&B scalar logging, optional generation metrics, and early stopping.


In [ ]:
trainer_callbacks = [
    WandbScalarLogger(
        avg_train_tokens_per_sample,
        gpu_total_memory_gb,
        CONFIG['target_gpu_hourly_cost_usd'],
    ),
    EarlyStoppingPatienceLogger(
        patience=CONFIG['early_stopping_patience'],
        threshold=CONFIG['early_stopping_threshold'],
    ),
    EarlyStoppingCallback(
        early_stopping_patience=CONFIG['early_stopping_patience'],
        early_stopping_threshold=CONFIG['early_stopping_threshold'],
    ),
]
if CONFIG['run_validation_generation']:
    trainer_callbacks.insert(
        1,
        ValidationGenerationLogger(
            validation_generation_rows,
            tokenizer,
            CONFIG['val_generation_log_examples'],
            CONFIG['target_gpu_hourly_cost_usd'],
        ),
    )

for callback in trainer_callbacks:
    print(callback.__class__.__name__)


### Create the SFT Trainer

At this point all ingredients are visible: model, datasets, arguments, and callbacks.


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train_dataset,
    eval_dataset=formatted_val_dataset,
    args=training_args,
    callbacks=trainer_callbacks,
)

print(f"Trainer train rows: {len(trainer.train_dataset):,}")
print(f"Trainer eval rows: {len(trainer.eval_dataset):,}")


### Mask Prompt Tokens

This Unsloth helper trains only on assistant labels. The system and user messages still condition the model, but they do not contribute to the loss.


In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)


### Show CUDA Memory Before Training

This gives us a concrete read on the GPU budget before the training loop starts.


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory_gb = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU: {gpu_stats.name}')
print(f'Max memory: {max_memory_gb} GB')
print(f'Reserved before training: {start_gpu_memory_gb} GB')


## 6. Train

The trainer evaluates validation loss periodically, early-stops when it stops improving, and restores the best checkpoint at the end.


In [ ]:
train_wall_start_time = time.perf_counter()
train_result = trainer.train()
train_wall_seconds = time.perf_counter() - train_wall_start_time

print(f"Training wall time: {train_wall_seconds / 60:.1f} minutes")
print(f"Trainer global step: {train_result.global_step:,}")
print(train_result.metrics)


### Log the Final Training Summary

Once training finishes, log final runtime, memory, and cost estimates to W&B.


In [ ]:
if wandb.run is not None:
    final_training_metrics = {
        f'train_result/{key}': value
        for key, value in train_result.metrics.items()
        if isinstance(value, (int, float))
    }
    if torch.cuda.is_available():
        final_peak_vram_gb = torch.cuda.max_memory_reserved() / 1024**3
        final_peak_vram_percent = 100 * final_peak_vram_gb / gpu_total_memory_gb
        final_training_metrics.update(
            {
                'train/wall_runtime_seconds': train_wall_seconds,
                'device_fit/final_peak_vram_reserved_gb': final_peak_vram_gb,
                'device_fit/final_peak_vram_reserved_percent': final_peak_vram_percent,
                'device_fit/final_vram_headroom_percent': 100 - final_peak_vram_percent,
                'cost/final_estimated_usd': (
                    train_wall_seconds / 3600 * CONFIG['target_gpu_hourly_cost_usd']
                ),
            }
        )
    wandb.log(final_training_metrics, step=train_result.global_step)

print("Final training summary logged to W&B.")


### Inspect the Best Checkpoint

`load_best_model_at_end=True` restores the checkpoint with the lowest validation loss.


In [ ]:
print(f'Best checkpoint: {trainer.state.best_model_checkpoint}')
print(f'Best validation loss: {trainer.state.best_metric}')
train_result


### Show CUDA Memory After Training

A small memory summary helps verify that the run stayed within the target GPU budget.


In [ ]:
end_gpu_memory_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_for_training_gb = round(end_gpu_memory_gb - start_gpu_memory_gb, 3)
print(f'Reserved after training: {end_gpu_memory_gb} GB')
print(f'Additional reserved during training: {used_for_training_gb} GB')


## 7. Final Test Accuracy

The test set is intentionally optional. During fast sweeps, validation loss is the run-selection signal; enable generation metrics and full test evaluation only for finalist runs.


In [ ]:
prediction_rows = []
correct = 0
test_accuracy = None

if CONFIG['run_test_evaluation']:
    for start in range(0, len(test_dataset), CONFIG['generation_eval_batch_size']):
        end = min(start + CONFIG['generation_eval_batch_size'], len(test_dataset))
        rows = row_batch(test_dataset, start, end)
        predictions = generate_label_batch(model, tokenizer, rows)
        for offset, (row, prediction) in enumerate(zip(rows, predictions)):
            index = start + offset
            normalized_prediction = normalize_label(prediction)
            exact_match = normalized_prediction == row['label_text']
            correct += int(exact_match)
            prediction_rows.append(
                {
                    'index': index,
                    'text': row['text'],
                    'expected_response': row['label_text'],
                    'prediction': prediction,
                    'normalized_prediction': normalized_prediction,
                    'exact_match': exact_match,
                }
            )
        if end % 256 == 0 or end == len(test_dataset):
            print(f'Evaluated {end:,}/{len(test_dataset):,} test rows')

    test_accuracy = correct / len(test_dataset)
    print(f'Test accuracy: {test_accuracy:.2%} ({correct:,}/{len(test_dataset):,})')
else:
    print("Skipped test evaluation. Set CONFIG['run_test_evaluation'] = True for finalist runs.")


### Inspect Final Test Predictions

When final evaluation runs, inspect a few concrete predictions before logging the table.


In [ ]:
if CONFIG['run_test_evaluation']:
    for row in prediction_rows[:20]:
        print('-' * 80)
        print(f"Text:        {row['text']}")
        print(f"Expected:    {row['expected_response']}")
        print(f"Prediction:  {row['prediction']}")
        print(f"Normalized:  {row['normalized_prediction']}")
        print(f"Match:       {row['exact_match']}")
else:
    print('No test predictions to inspect because final test evaluation was skipped.')


### Log Test Predictions to W&B

The prediction table lets us inspect misses directly in W&B, while scalar metrics make runs comparable.


In [ ]:
if wandb.run is None:
    wandb_run = wandb.init(
        project=CONFIG['wandb_project'],
        id=CONFIG['wandb_run_id'],
        name=wandb_run_name,
        resume='allow',
    )
    wandb_run_id = wandb_run.id

if CONFIG['run_test_evaluation']:
    predictions_table = wandb.Table(
        columns=[
            'index',
            'text',
            'expected_response',
            'prediction',
            'normalized_prediction',
            'exact_match',
        ]
    )
    for row in prediction_rows:
        predictions_table.add_data(
            row['index'],
            row['text'],
            row['expected_response'],
            row['prediction'],
            row['normalized_prediction'],
            row['exact_match'],
        )

    wandb.log({
        'test/predictions': predictions_table,
        'test/acc': test_accuracy,
        'test/correct': correct,
        'test/total': len(test_dataset),
        'best/val_loss': trainer.state.best_metric,
    })
    print(
        f"Logged {len(prediction_rows)} test predictions to W&B project "
        f"{CONFIG['wandb_project']} run {wandb_run_id}"
    )
else:
    wandb.log({
        'test/evaluation_skipped': 1,
        'best/val_loss': trainer.state.best_metric,
    })
    print('Logged that test evaluation was skipped.')


## 8. Save the LoRA Adapter

Saving only the adapter keeps the artifact small. Load it later on top of the same base model for inference or continued fine-tuning.


In [ ]:
adapter_dir = 'outputs/unsloth-banking77-generalization-lora'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Saved LoRA adapter to {adapter_dir}')


### Finish the W&B Run

Closing the run flushes the last metrics and prediction tables.


In [ ]:
if wandb.run is not None:
    wandb.finish()
    print('Finished W&B run.')


## Key Takeaways

- This notebook uses real Unsloth primitives and no fallback training path.
- The dataset source is `tsilva/banking77`: `system_prompt` -> system, `text` -> user, `label_text` -> assistant.
- The Hugging Face train split is split into train and validation partitions.
- Early stopping monitors validation loss and the best checkpoint is restored before test evaluation.
- Validation generation metrics are optional during training so fast sweeps can avoid expensive generation passes.
- Final test accuracy is optional and should be reserved for finalist configurations.
- W&B logs scalar training/eval metrics, throughput, device-fit, cost, and prediction tables.
